# BraTS 2023 preprocessing â€” corrected Windows protocol checks

Use this notebook instead of notebook 13. Select the **ncl_sae** kernel and **Run All**. It prepares slices and metadata; it does not start training. Existing completed patients can be resumed.

Fixes: Windows path case/slash equivalence, legacy notebook-wide hashes, correct protocol-check indentation, and path spelling inside inventory fingerprints. Actual changes to inputs, selected patients or preprocessing parameters still stop with a field-by-field explanation.

Run in the **ncl_sae** kernel, from top to bottom. This notebook prepares data; it does not train models.

Uses one scan per patient group in the labelled 2023 training archive. Groups are joined using the base BraTS-GLI ID and shared prior BraTS IDs in the mapping. Existing BraTS 2020 validation/test patients remain held out, using their original processed slices. All other selected groups become training data. The unlabelled challenge validation archive is excluded.

The old 37-patient validation and 37-patient test sets remain small. They have already been inspected during earlier work; this expansion is a follow-up study, not a newly untouched confirmatory test. Mapping-based grouping is not a guarantee against undocumented duplicate identities. The smallest scan suffix is selected deterministically; it is not assumed to indicate a clinically verified baseline scan.


In [ ]:
from pathlib import Path
import sys, json, hashlib, os, zipfile, xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import nibabel as nib
from skimage.transform import resize

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'scripts/revision_experiments.py').exists()), None)
assert ROOT is not None, 'Open this notebook within the project'
DATA = ROOT/'dataset/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData'
MAPPING = ROOT/'dataset/BraTS2023_2017_GLI_Mapping.xlsx'
OLD_MANIFEST = ROOT/'results/reviewer_revision_v2_lr5e4/split_manifest.csv'
OLD_METADATA = ROOT/'data/processed_flair/metadata.csv'
OUT = ROOT/'data/processed_flair_brats2023_expansion_v1'
AUDIT = ROOT/'results/brats2023_expansion_audit_v1'
RUN_PREPROCESS = True  # Run All will preprocess; no model training occurs.
IMG_SIZE = 128
MIN_BRAIN_FRACTION = 0.02
AUDIT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT/'scripts'))
from preprocess_brats import normalize_slice, dominant_label, LABEL_NAMES

def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024), b''): h.update(b)
    return h.hexdigest()

def atomic_json(path, value):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_suffix(path.suffix+'.tmp')
    temp.write_text(json.dumps(value,indent=2,allow_nan=False),encoding='utf-8')
    os.replace(temp,path)

def read_mapping(path):
    ns={'m':'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
    with zipfile.ZipFile(path) as z:
        strings=[''.join(x.itertext()) for x in ET.fromstring(z.read('xl/sharedStrings.xml')).findall('m:si',ns)]
        rows=[]
        for row in ET.fromstring(z.read('xl/worksheets/sheet1.xml')).findall('.//m:row',ns):
            values={}
            for c in row.findall('m:c',ns):
                v=c.find('m:v',ns)
                if v is not None:
                    values[''.join(filter(str.isalpha,c.attrib['r']))]=strings[int(v.text)] if c.get('t')=='s' else v.text
            rows.append(values)
    headers=rows[0]
    return pd.DataFrame(rows[1:]).rename(columns=headers).fillna('')


In [ ]:
mapping_all=read_mapping(MAPPING)
mapping=mapping_all[mapping_all.BraTS2023.str.startswith('BraTS-GLI-')].copy()
mapping_all[~mapping_all.BraTS2023.str.startswith('BraTS-GLI-')].to_csv(AUDIT/'mapping_rows_without_2023_id.csv',index=False)
assert not mapping.BraTS2023.duplicated().any()
old=pd.read_csv(OLD_MANIFEST)
old_patients=old[['patient_id','split']].drop_duplicates()
assert old_patients.patient_id.is_unique
old_split=dict(zip(old_patients.patient_id,old_patients.split))
# Union identities across all mapping rows, including scans outside this archive.
parent={}
def find(x):
    parent.setdefault(x,x)
    if parent[x]!=x: parent[x]=find(parent[x])
    return parent[x]
def union(a,b):
    a,b=find(a),find(b)
    if a!=b: parent[max(a,b)]=min(a,b)
valid=lambda s: str(s).strip() not in ('','N/A','nan','NA')
for r in mapping.to_dict('records'):
    scan=r['BraTS2023']; union(scan,'base:'+scan.rsplit('-',1)[0])
    for col in ['BraTS2021','BraTS2020','BraTS2019','BraTS2018','BraTS2017']:
        if valid(r.get(col,'')): union(scan,col+':'+r[col])
lookup=mapping.set_index('BraTS2023').to_dict('index')
flairs=sorted(DATA.rglob('*-t2f.nii.gz'))
assert flairs, 'No extracted training FLAIR files found'
records=[]
for flair in flairs:
    case=flair.name.removesuffix('-t2f.nii.gz')
    assert case in lookup, f'Missing mapping: {case}'
    mask=flair.with_name(case+'-seg.nii.gz')
    assert mask.is_file(), f'Missing mask: {case}'
    im,seg=nib.load(flair),nib.load(mask)
    assert len(im.shape)==3 and im.shape==seg.shape, case
    assert np.allclose(im.affine,seg.affine), case
    # Existing source slices use axial axis 2 in LPS orientation.
    assert nib.aff2axcodes(im.affine)==('L','P','S'), f'Unexpected orientation: {case}'
    records.append(dict(case_id=case,group_id=find(case),old_id=lookup[case].get('BraTS2020',''),
        flair_path=str(flair.resolve()),mask_path=str(mask.resolve()),
        flair_bytes=flair.stat().st_size,flair_mtime_ns=flair.stat().st_mtime_ns,
        mask_bytes=mask.stat().st_size,mask_mtime_ns=mask.stat().st_mtime_ns))
inventory=pd.DataFrame(records)
assert inventory.case_id.is_unique
# Protect held-out patients through any row sharing the same identity group.
group_old={}
for r in mapping.to_dict('records'):
    if r.get('BraTS2020') in old_split:
        group_old.setdefault(find(r['BraTS2023']),set()).add(r['BraTS2020'])
def assignment(g):
    splits={old_split[p] for p in group_old.get(g,set())}
    assert len(splits)<=1, f'Old split conflict for mapped patient group: {g}: {splits}'
    return next(iter(splits),'train')
inventory['split']=inventory.group_id.map(assignment)
inventory['selected']=False
# One scan per group. Held-out original metadata will be reused instead.
chosen=inventory[inventory.split=='train'].sort_values('case_id').drop_duplicates('group_id').index
inventory.loc[chosen,'selected']=True
mapped_old=set().union(*group_old.values()) if group_old else set()
unmatched=old_patients[~old_patients.patient_id.isin(mapped_old)]
assert not unmatched.split.isin(['val','test']).any(), 'Unresolved held-out identity: stop'
train=inventory[inventory.selected].copy()
assert train.group_id.is_unique
inventory.to_csv(AUDIT/'case_inventory.csv',index=False)
unmatched.to_csv(AUDIT/'unmatched_old_patients.csv',index=False)
summary={'labelled_cases':len(inventory),'patient_groups':inventory.group_id.nunique(),
 'selected_training_patients':len(train),'old_heldout_patients':old_patients[old_patients.split!='train'].groupby('split').size().to_dict(),
 'unmatched_old_training_patients':unmatched.patient_id.tolist(),
 'selection':'one smallest case ID per mapped training identity group; originals for old val/test'}
atomic_json(AUDIT/'inventory_summary.json',summary)
print(json.dumps(summary,indent=2))
display(inventory.groupby(['split','selected']).size().rename('cases').to_frame())


## Freeze the input selection
The protocol records mapping, old split, preprocessing source, and the inventory fingerprint. Re-running with changed inputs stops; choose a new versioned output directory for a changed experiment. Source images are hashed when each selected case is processed. Completed cases are resumed only after source-hash and output checks.


In [ ]:
import ntpath

def windows_path(value):
    """Canonical spelling of a Windows path, including drive case and slashes."""
    return ntpath.normcase(ntpath.normpath(str(value)))

def inventory_digest(frame):
    return hashlib.sha256(frame.to_csv(index=False).encode()).hexdigest()

canonical_inventory=inventory.copy()
for column in ['flair_path','mask_path']:
    canonical_inventory[column]=canonical_inventory[column].map(windows_path)

# Accept the original inventory hash only when it can be reproduced from the
# current complete inventory with a change of drive-letter spelling alone.
# Counts, identities, file sizes, timestamps and selection flags stay checked.
inventory_hashes={inventory_digest(inventory),inventory_digest(canonical_inventory)}
for upper in (True,False):
    variant=inventory.copy()
    for column in ['flair_path','mask_path']:
        def drive_case(value):
            text=str(value)
            return (text[0].upper() if upper else text[0].lower())+text[1:] if len(text)>1 and text[1]==':' else text
        variant[column]=variant[column].map(drive_case)
    inventory_hashes.add(inventory_digest(variant))

protocol={'version':'brats2023-expansion-v1','mapping_sha256':sha(MAPPING),
 'old_manifest_sha256':sha(OLD_MANIFEST),'old_metadata_sha256':sha(OLD_METADATA),
 'preprocessor_sha256':sha(ROOT/'scripts/preprocess_brats.py'),
 'inventory_sha256':inventory_digest(canonical_inventory),
 'img_size':IMG_SIZE,'min_brain_fraction':MIN_BRAIN_FRACTION,'label_remap':{'3':4},
 'normalization':'source normalize_slice, then skimage resize; axis 2 LPS',
 'summary':summary,'output':windows_path(OUT)}

def protocol_differences(stored,current,allowed_inventory_hashes):
    # This legacy hash covered runtime toggles and incidental cell edits.
    # All source-data hashes and scientific settings remain mandatory.
    ignored={'notebook_code_sha256'}
    saved={k:v for k,v in stored.items() if k not in ignored}
    active={k:v for k,v in current.items() if k not in ignored}
    if 'output' in saved: saved['output']=windows_path(saved['output'])
    if 'output' in active: active['output']=windows_path(active['output'])
    if saved.get('inventory_sha256') in allowed_inventory_hashes:
        saved['inventory_sha256']=active['inventory_sha256']
    return {key:{'saved':saved.get(key),'current':active.get(key)}
            for key in sorted(set(saved)|set(active)) if saved.get(key)!=active.get(key)}

protocol_file=AUDIT/'preprocessing_protocol.json'
if protocol_file.exists():
    stored=json.loads(protocol_file.read_text(encoding='utf-8'))
    differences=protocol_differences(stored,protocol,inventory_hashes)
    if differences:
        print(json.dumps(differences,indent=2))
        raise ValueError('Actual input, selection or preprocessing changes detected. See fields above; use new output/audit paths for a changed experiment.')
    print('Protocol verified. Windows path spelling and legacy notebook hashes handled.')
else:
    atomic_json(protocol_file,protocol)
    print('New preprocessing protocol recorded.')
OUT.mkdir(parents=True,exist_ok=True)

def preprocess_case(row, destination):
    destination=Path(destination);destination.mkdir(parents=True,exist_ok=True)
    case=row.case_id
    source={'flair_sha256':sha(row.flair_path),'mask_sha256':sha(row.mask_path)}
    done=destination/'complete.json'
    if done.exists():
        prior=json.loads(done.read_text())
        assert prior['source']==source, f'Source changed: {case}'
        frame=pd.read_csv(destination/'metadata.csv')
        for p in frame.file_path:
            obj=np.load(p,allow_pickle=True).item()
            assert obj['image'].shape==(IMG_SIZE,IMG_SIZE) and np.isfinite(obj['image']).all()
            assert obj['seg_mask'].shape==(IMG_SIZE,IMG_SIZE)
            assert set(np.unique(obj['seg_mask']))<={0,1,2,4}
        assert len(frame)==prior['slices']
        return frame
    flair=nib.load(row.flair_path).get_fdata()
    seg=nib.load(row.mask_path).get_fdata()
    assert np.isfinite(flair).all() and np.isfinite(seg).all(), case
    assert set(np.unique(seg))<={0,1,2,3}, f'Unexpected labels: {case}'
    seg[seg==3]=4
    rec=[]
    for z in range(flair.shape[2]):
        image=flair[:,:,z]
        if (image>0).mean()<MIN_BRAIN_FRACTION: continue
        image=resize(normalize_slice(image),(IMG_SIZE,IMG_SIZE),preserve_range=True,anti_aliasing=True).astype(np.float32)
        mask=resize(seg[:,:,z],(IMG_SIZE,IMG_SIZE),order=0,preserve_range=True,anti_aliasing=False).astype(np.uint8)
        label=dominant_label(mask)
        target=destination/f'{case}_slice{z:03d}.npy'
        with open(str(target)+'.tmp','wb') as f:
            np.save(f,{'image':image,'seg_mask':mask},allow_pickle=True)
        os.replace(str(target)+'.tmp',target)
        rec.append(dict(patient_id=row.group_id,case_id=case,slice_idx=z,file_path=str(target.resolve()),
          dominant_label=label,label_name=LABEL_NAMES[label],has_tumor=int(label!=0),split='train'))
    assert rec, f'No eligible slices: {case}'
    frame=pd.DataFrame(rec)
    frame.to_csv(destination/'metadata.csv',index=False)
    atomic_json(done,{'source':source,'slices':len(frame)})
    return frame


## Small real-data check
This processes one selected patient into a separate smoke directory. It does not create a training manifest. Inspect the image and mask before the full preprocessing cell.


In [ ]:
smoke=preprocess_case(next(train.itertuples(index=False)),AUDIT/'smoke_one_patient')
print('Smoke patient slices:',len(smoke))
import matplotlib.pyplot as plt
item=np.load(smoke.iloc[len(smoke)//2].file_path,allow_pickle=True).item()
fig,ax=plt.subplots(1,2,figsize=(7,3))
ax[0].imshow(item['image'],cmap='gray');ax[0].set_title('Normalised FLAIR')
ax[1].imshow(item['seg_mask'],vmin=0,vmax=4);ax[1].set_title('Mask after 3 â†’ 4 remap')
for a in ax:a.axis('off')
plt.show()


In [ ]:
if RUN_PREPROCESS:
    parts=[]
    for i,row in enumerate(train.itertuples(index=False),1):
        parts.append(preprocess_case(row,OUT/'train'/row.case_id))
        if i%10==0 or i==len(train):print(f'Processed/resumed {i}/{len(train)} training patients',flush=True)
    expanded_train=pd.concat(parts,ignore_index=True)
    prior=pd.read_csv(OLD_METADATA)
    held=prior[prior.split.isin(['val','test'])].copy()
    assert not held.empty
    expected=old[old.split.isin(['val','test'])]
    keys=['patient_id','slice_idx','split','dominant_label','has_tumor']
    sort=['patient_id','slice_idx']
    pd.testing.assert_frame_equal(held[keys].sort_values(sort).reset_index(drop=True),expected[keys].sort_values(sort).reset_index(drop=True),check_dtype=False)
    def resolve_old(r):
        p=Path(r.file_path)
        options=[p if p.is_absolute() else ROOT/p, OLD_METADATA.parent/r.split/p.name]
        match=next((p.resolve() for p in options if p.is_file()),None)
        assert match is not None,r.file_path
        return str(match)
    held['file_path']=held.apply(resolve_old,axis=1)
    held['case_id']=held.patient_id
    combined=pd.concat([expanded_train,held],ignore_index=True)
    assert combined.groupby('patient_id').split.nunique().max()==1
    assert not combined.duplicated(['patient_id','slice_idx']).any()
    combined.to_csv(OUT/'metadata.csv.tmp',index=False)
    os.replace(OUT/'metadata.csv.tmp',OUT/'metadata.csv')
    counts=combined.groupby('split').agg(patients=('patient_id','nunique'),slices=('slice_idx','size'))
    counts.to_csv(AUDIT/'processed_counts.csv')
    atomic_json(OUT/'ready.json',{'protocol_sha256':sha(protocol_file),'metadata_sha256':sha(OUT/'metadata.csv'),'counts':counts.to_dict('index')})
    display(counts)
    print('READY:',OUT/'metadata.csv')
else:
    print('Full preprocessing skipped. Set RUN_PREPROCESS=True and run this cell to prepare the expanded cohort.')


## Check compatibility with the existing training loader
This reads metadata and validates splits/paths. It does not initialise or train an encoder. The new configuration uses separate output directories so old results are preserved. Subsequent SAE training must use centering as in the completed replication; do not blindly run the old runner's entire SAE stage with its uncentered settings.


In [ ]:
if (OUT/'ready.json').exists():
    assert json.loads((OUT/'ready.json').read_text())['metadata_sha256']==sha(OUT/'metadata.csv')
    from revision_experiments import metadata as validate_metadata
    cfg=json.loads((ROOT/'scripts/experiment_config.json').read_text())
    cfg.update(experiment_name='brats2023_expansion_v1',modes=['ncl','simclr'],
       metadata=str((OUT/'metadata.csv').relative_to(ROOT)),output='results/brats2023_expansion_v1',root=str(ROOT))
    checked=validate_metadata(cfg)
    atomic_json(AUDIT/'encoder_config.json',cfg)
    print(f'Loader check passed: {len(checked):,} slices; no training started.')
    print('Encoder configuration:',AUDIT/'encoder_config.json')
else: print('Complete preprocessing first; training configuration is not ready yet.')
